# Clima 7 Dias — Juazeiro, BA
Estilo Google Weather  ·  Formato 1:1  ·  @reinanbr

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib.patheffects as pe
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from datetime import datetime, timedelta
from IPython.display import display, Image, clear_output
import ipywidgets as widgets
import warnings, io
warnings.filterwarnings('ignore')
print('imports OK')


imports OK


In [2]:
def to_dt(v):
    ts = (v - np.datetime64('1970-01-01T00:00')) / np.timedelta64(1, 's')
    return datetime.utcfromtimestamp(float(ts))

VARS_NEEDED = ['prmsl', 'prate', 'r2', 'cwat', 't2m']
N_DAYS = 7
BASE_DATE = datetime(2026, 4, 5)
DAYS = [BASE_DATE + timedelta(days=i) for i in range(N_DAYS)]
DAY_PT = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sab', 'Dom']

try:
    from noawclg.main import get_noaa_data
    print('Baixando GFS...')
    noaa = get_noaa_data(keys=VARS_NEEDED, date='20260405')
    print(noaa._ds)
    view = noaa.get_data_from_place('Juazeiro, Bahia, Brazil')
    ds   = view.dataset
    # Agrupa por dia (media, max, min)
    tv   = ds['time'].values
    _lat = float(ds['latitude'].values)
    _lon = float(ds['longitude'].values)
    # coleta arrays brutos
    _t2m_raw   = ds['t2m'].values.astype(float)
    if _t2m_raw.mean() > 100: _t2m_raw -= 273.15
    _prate_raw = ds['prate'].values.astype(float) * 3600
    _r2_raw    = ds['r2'].values.astype(float)
    _prmsl_raw = ds['prmsl'].values.astype(float)
    _cwat_raw  = ds['cwat'].values.astype(float)
    # agrega diariamente (assume step 3h, 8 pts/dia)
    def daily_agg(arr, n=N_DAYS, pts_day=8):
        return [arr[i*pts_day:(i+1)*pts_day] for i in range(n)]
    t2m_seg   = daily_agg(_t2m_raw)
    t_max  = np.array([s.max() for s in t2m_seg])
    t_min  = np.array([s.min() for s in t2m_seg])
    t_curve= _t2m_raw[:N_DAYS*8]
    prate  = np.array([daily_agg(_prate_raw)[i].mean() for i in range(N_DAYS)])
    r2val  = np.array([daily_agg(_r2_raw)[i].mean()    for i in range(N_DAYS)])
    prmsl  = np.array([daily_agg(_prmsl_raw)[i].mean() for i in range(N_DAYS)])
    cwat   = np.array([daily_agg(_cwat_raw)[i].mean()  for i in range(N_DAYS)])
    # condicoes baseadas em chuva + nuvens
    def classify(p, c):
        if p > 0.2: return 'chuva'
        if c > 0.3: return 'nublado'
        return 'ensolarado'
    conds = [classify(prate[i], cwat[i]) for i in range(N_DAYS)]
    DATA_SOURCE = 'GFS 0.25 / NOAA - NOMADS'
    print('GFS OK')
except Exception as e:
    print(f'Sintetico ({e})')
    np.random.seed(42)
    t_max  = np.array([32,32,32,33,32,33,33],dtype=float)
    t_min  = np.array([22,22,22,21,22,22,22],dtype=float)
    conds  = ['nublado','nublado','chuva','ensolarado','ensolarado','chuva','nublado']
    prate  = np.array([0.12,0.05,0.38,0.01,0.0,0.45,0.15])
    r2val  = np.array([76,70,82,55,48,80,72],dtype=float)
    prmsl  = np.array([1012.3,1011.1,1009.5,1013.2,1015.0,1010.7,1011.8])
    cwat   = np.array([0.35,0.28,0.52,0.12,0.08,0.48,0.31])
    def _mk(mn,mx):
        pts=[]
        for i in range(N_DAYS):
            for h in range(8):
                hr=h*3; frac=max(0,np.sin(max(0,hr-3)*np.pi/18))**1.5
                pts.append(mn[i]+(mx[i]-mn[i])*frac+np.random.normal(0,.25))
        return np.array(pts,dtype=float)
    t_curve = _mk(t_min,t_max)
    DATA_SOURCE = 'Dados sinteticos (exemplo)'
tx = np.linspace(0, N_DAYS, len(t_curve))
CURVE = {
    't2m':  t_curve,
    'prate':np.repeat(prate,8),
    'r2':   np.repeat(r2val,8),
    'prmsl':np.repeat(prmsl,8),
    'cwat': np.repeat(cwat,8),
}
DAILY = {'t2m':t_max,'prate':prate,'r2':r2val,'prmsl':prmsl,'cwat':cwat}
print('Dados prontos')


Baixando GFS...


KeyboardInterrupt: 

In [8]:
BG='#1C2331'; CARD='#243447'; CARD_HOD='#2C4060'
ORANGE='#FFA040'; BLUE_ICE='#64B5F6'; CLOUD='#8AAFC8'; RAIN='#5B8DB8'
TEXT1='#FFFFFF'; TEXT2='#9DB8D2'; TEXT3='#4A6080'; GR='#FFFFFF10'; HANDLE='#FF6B35'

VAR_STYLES = {
    't2m':  {'line':'#D4A820','label':'Temperatura',       'unit':'C',    'yfmt':'%.0f','extra':'Max e minima diaria'},
    'prate':{'line':'#64B5F6','label':'Precipitacao',      'unit':'mm/h', 'yfmt':'%.2f','extra':'Taxa media diaria'},
    'r2':   {'line':'#66BB6A','label':'Humidade Relativa', 'unit':'%',    'yfmt':'%.0f','extra':'Humidade relativa a 2 m'},
    'prmsl':{'line':'#BA68C8','label':'Pressao Atmosferica','unit':'hPa', 'yfmt':'%.1f','extra':'Pressao ao nivel do mar'},
    'cwat': {'line':'#90A4AE','label':'Agua em Nuvens',    'unit':'kg/m2','yfmt':'%.2f','extra':'Coluna total de agua'},
}

def draw_icon(ax, x, y, cond, s=0.40):
    if cond == 'ensolarado':
        ax.add_patch(mpatches.Circle((x,y), s*.42, color=ORANGE, zorder=12, lw=0))
        for ang in np.linspace(0,2*np.pi,8,endpoint=False):
            ax.plot([x+np.cos(ang)*s*.57, x+np.cos(ang)*s*.80],
                    [y+np.sin(ang)*s*.57, y+np.sin(ang)*s*.80],
                    color=ORANGE, lw=2.0, zorder=11, solid_capstyle='round')
    elif cond == 'nublado':
        ax.add_patch(mpatches.Circle((x-.14*s,y+.18*s),s*.24,color='#5272A0',zorder=10,lw=0))
        ax.add_patch(mpatches.Circle((x-.06*s,y+.19*s),s*.18,color=CLOUD,zorder=11,lw=0))
        for cx2,cy2,cr in [(x-.16,y+.01,s*.29),(x+.11,y+.09,s*.23),(x+.09,y-.05,s*.19),(x-.05,y-.09,s*.21)]:
            ax.add_patch(mpatches.Circle((cx2,cy2),cr,color=CLOUD,zorder=12,lw=0))
    elif cond == 'chuva':
        for cx2,cy2,cr in [(x-.14,y+.10,s*.27),(x+.09,y+.16,s*.21),(x+.07,y+.03,s*.17),(x-.04,y-.01,s*.19)]:
            ax.add_patch(mpatches.Circle((cx2,cy2),cr,color=RAIN,zorder=12,lw=0))
        for gx,gy in [(x-.20,y-.16),(x-.02,y-.22),(x+.16,y-.16)]:
            ax.plot([gx,gx-.05],[gy,gy-.14],color=BLUE_ICE,lw=2.0,zorder=13,solid_capstyle='round')

print('Tema e icones OK')


Tema e icones OK


In [4]:
def build_plot(key, dpi_val=150):
    st  = VAR_STYLES[key]
    dat = CURVE[key]
    lc  = st['line']
    lo  = float(dat.min()); hi = float(dat.max()); span = hi - lo
    if span < 1e-6: span = 1.0

    fig = matplotlib.figure.Figure(figsize=(12,12), dpi=dpi_val)
    FigureCanvasAgg(fig)
    fig.patch.set_facecolor(BG)

    def T(x,y,s,**kw): fig.text(x,y,s,**kw)

    # ── HEADER ──────────────────────────────────────────────────────────
    T(.06,.957,'Juazeiro, BA',color=TEXT1,fontsize=22,fontweight='black',va='top',
      path_effects=[pe.withStroke(linewidth=4,foreground=BG)])
    T(.06,.912,f'{st["label"]}  |  Proximos 7 dias',color=TEXT2,fontsize=11,
      fontfamily='monospace',va='top')
    T(.06,.877,st['extra'],color=TEXT3,fontsize=8.5,fontfamily='monospace',va='top')
    fig.add_artist(mlines.Line2D([.06,.94],[.864,.864],transform=fig.transFigure,
                                  color=lc,linewidth=2,alpha=.75))
    # valor hoje
    dv0 = DAILY[key][0]
    if key == 't2m':
        val_txt = f'{t_max[0]:.0f} C'; sub_txt = f'min {t_min[0]:.0f} C'
    elif key == 'prmsl':
        val_txt = f'{dv0:.1f} {st["unit"]}'; sub_txt = ''
    elif key == 'r2':
        val_txt = f'{dv0:.0f} {st["unit"]}'; sub_txt = ''
    else:
        val_txt = f'{dv0:.2f} {st["unit"]}'; sub_txt = ''
    T(.94,.957,val_txt,color=TEXT1,fontsize=28,fontweight='black',va='top',ha='right',
      path_effects=[pe.withStroke(linewidth=5,foreground=BG)])
    if sub_txt:
        T(.94,.908,sub_txt,color=TEXT2,fontsize=11,va='top',ha='right',fontfamily='monospace')

    # icone grande hoje
    aib = fig.add_axes([.83,.855,.09,.09])
    aib.set_xlim(-1,1); aib.set_ylim(-1,1); aib.axis('off')
    draw_icon(aib,0,0,conds[0],s=1.6)

    # ── GRAFICO ─────────────────────────────────────────────────────────
    ax = fig.add_axes([.06,.36,.88,.475])
    ax.set_facecolor(BG)
    ax.set_xlim(0,N_DAYS)
    ax.set_ylim(lo-span*.30, hi+span*.35)
    for d in range(N_DAYS+1):
        ax.axvline(d,color=GR,linewidth=.8,zorder=0)
    ti = np.linspace(0,N_DAYS,600); ci = np.interp(ti,tx,dat)
    ax.fill_between(ti,lo-span*.40,ci,color=lc,alpha=.12,zorder=2,linewidth=0)
    ax.plot(tx,dat,color=lc,lw=3.5,zorder=6,solid_capstyle='round')
    ax.plot(tx,dat,color=lc,lw=9,alpha=.12,zorder=5,solid_capstyle='round')
    ax.yaxis.set_major_locator(mticker.MaxNLocator(5))
    ax.grid(axis='y',color=GR,linewidth=.7,zorder=0)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(st['yfmt']))
    ax.tick_params(axis='y',colors=TEXT3,labelsize=8.5,length=0,pad=4)
    ax.tick_params(axis='x',length=0,pad=6)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_ylabel(st['unit'],color=TEXT3,fontsize=9,fontfamily='monospace',labelpad=6)
    ax.set_xticks([d+.5 for d in range(N_DAYS)])
    ax.set_xticklabels([DAY_PT[d]+'\n'+DAYS[d].strftime('%d/%m') for d in range(N_DAYS)],
                       color=TEXT2,fontsize=8.5,fontfamily='monospace')

    for d in range(N_DAYS):
        xc = d+0.5; dv = DAILY[key][d]
        yv = float(np.mean(dat[d*8:(d+1)*8]))
        ax.scatter(xc,yv,s=55,color=lc,zorder=10,edgecolors='white',lw=1.2)
        if key == 't2m':
            ax.text(xc,t_max[d]+span*.11,f'{t_max[d]:.0f}',ha='center',va='bottom',
                    color=TEXT1,fontsize=9.5,fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=2.5,foreground=BG)])
            ax.text(xc,t_min[d]-span*.14,f'{t_min[d]:.0f}',ha='center',va='top',
                    color=TEXT2,fontsize=8.5,
                    path_effects=[pe.withStroke(linewidth=2,foreground=BG)])
        else:
            lbl = (f'{dv:.1f}' if key=='prmsl' else f'{dv:.0f}' if key=='r2' else f'{dv:.2f}')
            ax.text(xc,yv+span*.13,lbl,ha='center',va='bottom',
                    color=TEXT1,fontsize=9.5,fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=2.5,foreground=BG)])

    # ── CARDS DIAS ──────────────────────────────────────────────────────
    card_h=.278; card_y=.060; cw=.88/N_DAYS; cx0=.06
    for d in range(N_DAYS):
        is_t=(d==0); cx=cx0+d*cw; cc=CARD_HOD if is_t else CARD
        rect=FancyBboxPatch((cx+.005,card_y),cw-.010,card_h-.010,
                             boxstyle='round,pad=0.006',
                             facecolor=cc,edgecolor=lc if is_t else '#FFFFFF18',
                             linewidth=1.8 if is_t else .6,
                             transform=fig.transFigure,zorder=3,clip_on=False)
        fig.add_artist(rect)
        T(cx+cw/2,card_y+card_h-.030,DAY_PT[d],
          color=TEXT1 if is_t else TEXT2,fontsize=9.5,
          fontweight='bold' if is_t else 'normal',
          ha='center',va='top',zorder=4)
        T(cx+cw/2,card_y+card_h-.058,DAYS[d].strftime('%d/%m'),
          color=TEXT3,fontsize=7.5,fontfamily='monospace',
          ha='center',va='top',zorder=4)
        ia=fig.add_axes([cx+cw*.14,card_y+card_h*.28,cw*.72,card_h*.44])
        ia.set_xlim(-1,1); ia.set_ylim(-1,1); ia.axis('off'); ia.set_facecolor('none')
        draw_icon(ia,0,0,conds[d],s=1.25)
        dv=DAILY[key][d]
        if key=='t2m':
            vlbl=f'{t_max[d]:.0f} / {t_min[d]:.0f} C'
        elif key=='prmsl':
            vlbl=f'{dv:.1f} {st["unit"]}'
        elif key=='r2':
            vlbl=f'{dv:.0f} {st["unit"]}'
        else:
            vlbl=f'{dv:.2f} {st["unit"]}'
        T(cx+cw/2,card_y+.018,vlbl,
          color=TEXT1 if is_t else TEXT2,
          fontsize=9.5 if is_t else 9,
          fontweight='bold' if is_t else 'normal',
          ha='center',va='bottom',zorder=4,
          path_effects=[pe.withStroke(linewidth=2,foreground=cc)])

    # ── RODAPE ──────────────────────────────────────────────────────────
    fig.add_artist(mlines.Line2D([.06,.94],[.042,.042],transform=fig.transFigure,
                                  color=GR,linewidth=.8))
    T(.06,.032,f'Fonte: {DATA_SOURCE}',color=TEXT3,fontsize=7,fontfamily='monospace',va='top')
    T(.94,.032,'@reinanbr',color=HANDLE,fontsize=13,fontweight='black',
      fontfamily='monospace',va='top',ha='right',
      path_effects=[pe.withStroke(linewidth=4,foreground=BG)])

    buf=io.BytesIO()
    fig.savefig(buf,format='png',dpi=dpi_val,bbox_inches='tight',facecolor=BG,edgecolor='none')
    buf.seek(0)
    display(Image(data=buf.read()))
    if salvar:
        fname=f'juazeiro_{key}_7d.png'; buf.seek(0)
        with open(fname,'wb') as f: f.write(buf.read())
        print(f'Salvo: {fname}')

print('build_plot() OK')


build_plot() OK


In [5]:
w_vars = widgets.SelectMultiple(
    options=[
        ('Temperatura (t2m)',       't2m'),
        ('Precipitacao (prate)',    'prate'),
        ('Humidade (r2)',           'r2'),
        ('Pressao Atm. (prmsl)',   'prmsl'),
        ('Agua em Nuvens (cwat)',  'cwat'),
    ],
    value=['t2m','prate','r2','prmsl','cwat'],
    description='Variaveis:', rows=5,
    layout=widgets.Layout(width='340px'),
    style={'description_width':'80px'},
)
w_dpi = widgets.RadioButtons(
    options=[('Previa (100 dpi)',100),('Alta (150 dpi)',150),('Maxima (250 dpi)',250)],
    value=150, description='Qualidade:',
    style={'description_width':'80px'},
    layout=widgets.Layout(margin='4px 0'),
)
w_salvar_cb = widgets.Checkbox(value=False, description='Salvar PNGs no disco',
    indent=False, layout=widgets.Layout(margin='6px 0'))
w_btn = widgets.Button(description='Gerar Plots', button_style='success',
    layout=widgets.Layout(width='148px',height='40px',margin='10px 0'))
w_out = widgets.Output()

def on_click(b):
    global salvar
    sel = list(w_vars.value)
    if not sel:
        with w_out: clear_output(); print('Selecione ao menos uma variavel.')
        return
    salvar = w_salvar_cb.value
    with w_out:
        clear_output(wait=True)
        print(f'Gerando {len(sel)} plot(s) estilo Google Weather -- {w_dpi.value} dpi...')
        for k in sel:
            build_plot(k, dpi_val=w_dpi.value)
        print('Concluido!')

salvar = False
w_btn.on_click(on_click)
painel = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 10px">Clima 7 Dias - Juazeiro BA - estilo Google Weather</h3>'),
    widgets.HBox([
        widgets.VBox([w_vars,
            widgets.HTML('<span style="color:#888;font-size:11px">Ctrl/Cmd+Click para multiplas.</span>')]),
        widgets.VBox([w_dpi, w_salvar_cb, w_btn]),
    ], layout=widgets.Layout(gap='28px',align_items='flex-start')),
])
display(painel, w_out)


Output()